# BCS Determination — Visualisation des résultats

Ce notebook permet de :
1. **Visualiser les courbes d'entraînement** depuis les logs TensorBoard
2. **Charger le meilleur checkpoint** (79% val accuracy, epoch 15)
3. **Évaluer sur le jeu de test** avec matrice de confusion, rapport par classe
4. **Visualiser des prédictions** (correctes et incorrectes)

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/home/SPeillet/bcs_determination")
sys.path.insert(0, str(ROOT / "src"))

import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from PIL import Image
from collections import defaultdict

import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from bcs_pipeline.lightning_module.bcs_determination_module import LitBcsDetermination
from bcs_pipeline.data.stanford_bcs_datamodule import StanfordBcsDataModule

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

In [ ]:
EXPERIMENT_DIR = ROOT / "experiments" / "resnet50_adam_cosine_annealing"
CHECKPOINT_PATH = EXPERIMENT_DIR / "checkpoints" / "epoch=epoch=15-val_acc=val_acc=0.79-step=8240.ckpt"
TB_LOG_DIR = EXPERIMENT_DIR / "tensorboard" / "bcs_determination" / "version_3"
DATA_DIR = ROOT / "data" / "stanford_dogs"
SPLIT_DIR = EXPERIMENT_DIR / "splits"
STATS_PATH = EXPERIMENT_DIR / "stats" / "dataset_stats.json"

MODEL_NAME = "resnet50"
NUM_CLASSES = 120
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 8
SEED = 42

print(f"Checkpoint: {CHECKPOINT_PATH.name}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"TensorBoard log dir exists: {TB_LOG_DIR.exists()}")
print(f"Data dir exists: {DATA_DIR.exists()}")

## 2. Courbes d'entraînement (TensorBoard)

In [ ]:
def read_tb_scalars(tb_dir, tags):
    ea = EventAccumulator(str(tb_dir), size_guidance={"scalars": 99999})
    ea.Reload()
    data = {}
    for tag in tags:
        if tag in ea.Tags()["scalars"]:
            events = ea.Scalars(tag)
            data[tag] = {
                "steps": [e.step for e in events],
                "values": [e.value for e in events],
                "wall_time": [e.wall_time for e in events],
            }
    return data

TAGS = [
    "train/loss_epoch", "train/acc_epoch", "train/acc_top5",
    "val/loss", "val/acc", "val/acc_top5", "val/acc_best",
    "lr-AdamW",
    "monitor/overfit_gap", "monitor/generalization_gap",
    "test/loss", "test/acc", "test/acc_top5",
]

tb_data = read_tb_scalars(TB_LOG_DIR, TAGS)
print(f"Loaded {len(tb_data)} scalar tags from TensorBoard")
for tag, d in tb_data.items():
    print(f"  {tag}: {len(d['values'])} points")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Loss ---
ax = axes[0, 0]
if "train/loss_epoch" in tb_data:
    d = tb_data["train/loss_epoch"]
    ax.plot(d["steps"], d["values"], label="Train Loss", color="#1f77b4")
if "val/loss" in tb_data:
    d = tb_data["val/loss"]
    ax.plot(d["steps"], d["values"], label="Val Loss", color="#ff7f0e")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Loss (Cross-Entropy with Label Smoothing)")
ax.legend()

# --- Accuracy ---
ax = axes[0, 1]
if "train/acc_epoch" in tb_data:
    d = tb_data["train/acc_epoch"]
    ax.plot(d["steps"], d["values"], label="Train Acc (Top-1)", color="#1f77b4")
if "val/acc" in tb_data:
    d = tb_data["val/acc"]
    ax.plot(d["steps"], d["values"], label="Val Acc (Top-1)", color="#ff7f0e")
if "val/acc_best" in tb_data:
    d = tb_data["val/acc_best"]
    ax.plot(d["steps"], d["values"], label="Val Best Acc", color="#2ca02c", linestyle="--")
if "train/acc_top5" in tb_data:
    d = tb_data["train/acc_top5"]
    ax.plot(d["steps"], d["values"], label="Train Acc (Top-5)", color="#1f77b4", alpha=0.4, linestyle=":")
if "val/acc_top5" in tb_data:
    d = tb_data["val/acc_top5"]
    ax.plot(d["steps"], d["values"], label="Val Acc (Top-5)", color="#ff7f0e", alpha=0.4, linestyle=":")
ax.set_xlabel("Step")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy (Top-1 & Top-5)")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)

# --- Learning Rate ---
ax = axes[1, 0]
if "lr-AdamW" in tb_data:
    d = tb_data["lr-AdamW"]
    ax.plot(d["steps"], d["values"], color="#d62728")
ax.set_xlabel("Step")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate (Cosine Annealing)")
ax.ticklabel_format(axis="y", style="scientific", scilimits=(0, 0))

# --- Overfitting / Generalization Gap ---
ax = axes[1, 1]
if "monitor/overfit_gap" in tb_data:
    d = tb_data["monitor/overfit_gap"]
    ax.plot(d["steps"], d["values"], label="Overfit Gap (val_loss - train_loss)", color="#9467bd")
if "monitor/generalization_gap" in tb_data:
    d = tb_data["monitor/generalization_gap"]
    ax.plot(d["steps"], d["values"], label="Gen Gap (train_acc - val_acc)", color="#8c564b")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Step")
ax.set_ylabel("Gap")
ax.set_title("Overfitting & Generalization Gap")
ax.legend()

fig.suptitle("Training Metrics — ResNet50 on Stanford Dogs (120 breeds)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
if "test/acc" in tb_data:
    print(f"Test Accuracy (Top-1): {tb_data['test/acc']['values'][-1]:.4f}")
if "test/acc_top5" in tb_data:
    print(f"Test Accuracy (Top-5): {tb_data['test/acc_top5']['values'][-1]:.4f}")
if "test/loss" in tb_data:
    print(f"Test Loss: {tb_data['test/loss']['values'][-1]:.4f}")
if "val/acc_best" in tb_data:
    print(f"Best Val Accuracy (Top-1): {max(tb_data['val/acc_best']['values']):.4f}")
    print(f"Best Val Accuracy (Top-5): {max(tb_data['val/acc_top5']['values']):.4f}")

## 3. Chargement du modèle et des données

In [ ]:
checkpoint = torch.load(str(CHECKPOINT_PATH), map_location=device, weights_only=False)
ckpt_state = checkpoint["state_dict"]

has_sequential_fc = any("net.backbone.fc.1" in k for k in ckpt_state.keys())
dropout_val = 0.3 if has_sequential_fc else 0.0
print(f"Checkpoint fc format: {'Sequential(Dropout+Linear)' if has_sequential_fc else 'plain Linear'}")

model = LitBcsDetermination(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    regularization={"dropout": dropout_val, "stochastic_depth": 0.0},
)

model_state = model.state_dict()
new_state = {}
for key, val in ckpt_state.items():
    if key in model_state:
        new_state[key] = val
    elif key == "net.backbone.fc.weight":
        new_state["net.backbone.fc.1.weight"] = val
    elif key == "net.backbone.fc.bias":
        new_state["net.backbone.fc.1.bias"] = val
    else:
        new_state[key] = val

missing, unexpected = model.load_state_dict(new_state, strict=False)
if missing:
    print(f"Missing keys: {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")

model.to(device)
model.eval()
print(f"Model loaded from: {CHECKPOINT_PATH.name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
data_module = StanfordBcsDataModule(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE,
    val_split=0.1,
    test_split=0.1,
    seed=SEED,
    split_dir=str(SPLIT_DIR),
)
data_module.prepare_data()
data_module.setup()

test_loader = data_module.test_dataloader()
print(f"Test set size: {len(data_module.data_test)}")
print(f"Val set size: {len(data_module.data_val)}")
print(f"Train set size: {len(data_module.data_train)}")

class_names_raw = data_module.classes
class_names = ["-".join(c.split("-")[1:]) for c in class_names_raw]
print(f"Number of classes: {len(class_names)}")
print(f"Example classes: {class_names[:5]}")

## 4. Évaluation complète sur le jeu de test

In [ ]:
all_preds = []
all_labels = []
all_probs = []
all_images = []

full_dataset = data_module.data_test.subset.dataset
test_indices = data_module.test_indices

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images_dev = images.to(device)
        logits = model(images_dev)
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

correct = all_preds == all_labels
accuracy = correct.mean()

top5_preds = np.argsort(all_probs, axis=1)[:, ::-1][:, :5]
top5_correct = np.array([label in top5 for top5, label in zip(top5_preds, all_labels)])
top5_accuracy = top5_correct.mean()

print(f"Test Top-1 Accuracy: {accuracy:.4f} ({correct.sum()}/{len(all_labels)})")
print(f"Test Top-5 Accuracy: {top5_accuracy:.4f} ({top5_correct.sum()}/{len(all_labels)})")

### 4.1 Matrice de confusion

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, cmap="Blues", ax=ax, xticklabels=class_names, yticklabels=class_names)
ax.set_xlabel("Predicted", fontsize=14)
ax.set_ylabel("True", fontsize=14)
ax.set_title(f"Confusion Matrix — Test Accuracy: {accuracy:.2%}", fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "confusion_matrix_full.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.2 Matrice de confusion — classes les plus confondues

In [ ]:
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)

confusion_pairs = []
for true_cls in range(NUM_CLASSES):
    for pred_cls in range(NUM_CLASSES):
        if cm_no_diag[true_cls, pred_cls] > 0:
            confusion_pairs.append((true_cls, pred_cls, cm_no_diag[true_cls, pred_cls]))

confusion_pairs.sort(key=lambda x: x[2], reverse=True)

top_n = 20
confused_classes = set()
for true_cls, pred_cls, count in confusion_pairs[:top_n]:
    confused_classes.add(true_cls)
    confused_classes.add(pred_cls)
confused_classes = sorted(confused_classes)

cm_subset = cm[np.ix_(confused_classes, confused_classes)]
subset_names = [class_names[i] for i in confused_classes]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_subset, annot=True, fmt="d", cmap="Reds",
    xticklabels=subset_names, yticklabels=subset_names, ax=ax,
)
ax.set_xlabel("Predicted", fontsize=14)
ax.set_ylabel("True", fontsize=14)
ax.set_title(f"Top {top_n} Confused Class Pairs", fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "confusion_matrix_top_confused.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTop {top_n} most confused class pairs:")
for i, (true_cls, pred_cls, count) in enumerate(confusion_pairs[:top_n]):
    print(f"  {i+1:2d}. {class_names[true_cls]:30s} <-> {class_names[pred_cls]:30s} : {count} samples")

### 4.3 Accuracy par classe

In [ ]:
per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
for label, pred in zip(all_labels, all_preds):
    per_class_total[label] += 1
    if label == pred:
        per_class_correct[label] += 1

per_class_acc = {}
for cls in range(NUM_CLASSES):
    total = per_class_total.get(cls, 0)
    if total > 0:
        per_class_acc[cls] = per_class_correct.get(cls, 0) / total
    else:
        per_class_acc[cls] = 0.0

sorted_classes = sorted(per_class_acc.items(), key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(18, 8))
x_pos = np.arange(NUM_CLASSES)
accs = [per_class_acc[i] for i in range(NUM_CLASSES)]
colors = ["#2ca02c" if a >= 0.8 else "#ff7f0e" if a >= 0.5 else "#d62728" for a in accs]
ax.bar(x_pos, accs, color=colors, edgecolor="none")
ax.set_xticks(x_pos)
ax.set_xticklabels([class_names[i] for i in range(NUM_CLASSES)], rotation=90, fontsize=5)
ax.set_ylabel("Accuracy")
ax.set_title(f"Per-Class Accuracy on Test Set (mean={accuracy:.2%})")
ax.axhline(accuracy, color="black", linestyle="--", alpha=0.5, label=f"Mean = {accuracy:.2%}")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "per_class_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("=== TOP 10 Best Classes ===")
for cls, acc in sorted_classes[-10:][::-1]:
    print(f"  {class_names[cls]:30s} : {acc:.2%} ({per_class_correct.get(cls, 0)}/{per_class_total.get(cls, 0)})")

print("\n=== TOP 10 Worst Classes ===")
for cls, acc in sorted_classes[:10]:
    print(f"  {class_names[cls]:30s} : {acc:.2%} ({per_class_correct.get(cls, 0)}/{per_class_total.get(cls, 0)})")

### 4.4 Rapport de classification détaillé

In [ ]:
report = classification_report(
    all_labels, all_preds,
    target_names=class_names,
    zero_division=0,
)
print(report)

report_dict = classification_report(
    all_labels, all_preds,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T.drop(index=["macro avg", "weighted avg", "accuracy"], errors="ignore")
report_df = report_df.sort_values("f1-score", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(20, 22))
for ax, metric, color in zip(axes, ["precision", "recall", "f1-score"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    vals = report_df[metric].values
    ax.barh(range(len(vals)), vals, color=color, alpha=0.8)
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(report_df.index, fontsize=5)
    ax.set_xlim(0, 1.05)
    ax.set_title(metric.capitalize())
    ax.axvline(report_df[metric].mean(), color="black", linestyle="--", alpha=0.5)

plt.suptitle("Per-Class Precision / Recall / F1-Score", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "classification_report.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Visualisation des prédictions

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


def show_predictions(images, labels, preds, probs, n=16, title="Predictions"):
    """Display a grid of images with predicted vs true labels."""
    n = min(n, len(images))
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    if rows == 1:
        axes = [axes] if cols == 1 else axes
    axes_flat = axes.flatten() if rows > 1 else axes

    for i in range(n):
        ax = axes_flat[i]
        img = denormalize(images[i].cpu())
        ax.imshow(img)
        pred_name = class_names[preds[i]]
        true_name = class_names[labels[i]]
        conf = probs[i, preds[i]]
        is_correct = preds[i] == labels[i]
        color = "green" if is_correct else "red"
        symbol = "\u2713" if is_correct else "\u2717"
        ax.set_title(f"{symbol} {pred_name}\n({conf:.1%}) [True: {true_name}]", fontsize=9, color=color)
        ax.axis("off")

    for i in range(n, len(axes_flat)):
        axes_flat[i].axis("off")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 5.1 Prédictions correctes (haute confiance)

In [ ]:
correct_idx = np.where(correct)[0]
correct_confidences = all_probs[correct_idx, all_preds[correct_idx]]
sorted_by_conf = correct_idx[np.argsort(correct_confidences)[::-1]]

images_list = []
labels_list = []
preds_list = []
probs_list = []

for idx in sorted_by_conf[:16]:
    dataset_idx = test_indices[idx]
    img_tensor, label = data_module.data_test[idx]
    images_list.append(img_tensor)
    labels_list.append(all_labels[idx])
    preds_list.append(all_preds[idx])
    probs_list.append(all_probs[idx])

images_batch = torch.stack(images_list)
labels_arr = np.array(labels_list)
preds_arr = np.array(preds_list)
probs_arr = np.array(probs_list)

show_predictions(images_batch, labels_arr, preds_arr, probs_arr, n=16,
                title="Correct Predictions (Highest Confidence)")

### 5.2 Prédictions incorrectes

In [ ]:
incorrect_idx = np.where(~correct)[0]
incorrect_conf = all_probs[incorrect_idx, all_preds[incorrect_idx]]
sorted_incorrect = incorrect_idx[np.argsort(incorrect_conf)[::-1]]

images_list = []
labels_list = []
preds_list = []
probs_list = []

for idx in sorted_incorrect[:16]:
    img_tensor, label = data_module.data_test[idx]
    images_list.append(img_tensor)
    labels_list.append(all_labels[idx])
    preds_list.append(all_preds[idx])
    probs_list.append(all_probs[idx])

images_batch = torch.stack(images_list)
labels_arr = np.array(labels_list)
preds_arr = np.array(preds_list)
probs_arr = np.array(probs_list)

show_predictions(images_batch, labels_arr, preds_arr, probs_arr, n=16,
                title="Incorrect Predictions (Highest Confidence)")

### 5.3 Distribution des confiances

In [ ]:
max_probs = all_probs[np.arange(len(all_preds)), all_preds]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.hist(max_probs[correct], bins=50, alpha=0.7, label="Correct", color="green", density=True)
ax.hist(max_probs[~correct], bins=50, alpha=0.7, label="Incorrect", color="red", density=True)
ax.set_xlabel("Confidence (max probability)")
ax.set_ylabel("Density")
ax.set_title("Confidence Distribution")
ax.legend()

ax = axes[1]
ax.hist(max_probs[correct], bins=50, alpha=0.7, label="Correct", color="green", cumulative=True, density=True)
ax.hist(max_probs[~correct], bins=50, alpha=0.7, label="Incorrect", color="red", cumulative=True, density=True)
ax.set_xlabel("Confidence (max probability)")
ax.set_ylabel("Cumulative Density")
ax.set_title("Cumulative Confidence Distribution")
ax.legend()

plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean confidence (correct):   {max_probs[correct].mean():.3f}")
print(f"Mean confidence (incorrect): {max_probs[~correct].mean():.3f}")
print(f"Min confidence (correct):    {max_probs[correct].min():.3f}")
print(f"Max confidence (incorrect):  {max_probs[~correct].max():.3f}")

### 5.4 Calibration du modèle (Reliability Diagram)

In [ ]:
n_bins = 15
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_accs = []
bin_confs = []
bin_counts = []

for i in range(n_bins):
    mask = (max_probs >= bin_edges[i]) & (max_probs < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_accs.append(correct[mask].mean())
        bin_confs.append(max_probs[mask].mean())
        bin_counts.append(mask.sum())
    else:
        bin_accs.append(0)
        bin_confs.append(bin_centers[i])
        bin_counts.append(0)

bin_accs = np.array(bin_accs)
bin_confs = np.array(bin_confs)
bin_counts = np.array(bin_counts)

ece = np.sum((bin_counts / bin_counts.sum()) * np.abs(bin_accs - bin_confs))

fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(bin_centers, bin_accs, width=1 / n_bins * 0.9, alpha=0.7, color="#1f77b4", edgecolor="black",
       label="Accuracy in bin")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Confidence")
ax.set_ylabel("Accuracy")
ax.set_title(f"Reliability Diagram (ECE = {ece:.4f})")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "reliability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.5 Visualisation aléatoire d'images du jeu de test

In [ ]:
np.random.seed(SEED)
random_test_idx = np.random.choice(len(test_indices), size=16, replace=False)

images_list = []
labels_list = []
preds_list = []
probs_list = []

for idx in random_test_idx:
    img_tensor, label = data_module.data_test[idx]
    images_list.append(img_tensor)
    labels_list.append(all_labels[idx])
    preds_list.append(all_preds[idx])
    probs_list.append(all_probs[idx])

images_batch = torch.stack(images_list)
show_predictions(
    images_batch,
    np.array(labels_list), np.array(preds_list), np.array(probs_list),
    n=16, title="Random Test Samples",
)

## 6. Embedding visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

hooks = []
features = []

def hook_fn(module, input, output):
    if isinstance(output, torch.Tensor):
        features.append(output.mean(dim=[2, 3]).detach().cpu())

if hasattr(model.net, "backbone"):
    layer4 = model.net.backbone.layer4
else:
    layer4 = model.net.layer4
hooks.append(layer4.register_forward_hook(hook_fn))

all_features = []
feat_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        features.clear()
        _ = model(images.to(device))
        if features:
            all_features.append(features[0])
            feat_labels.extend(labels.numpy())

for h in hooks:
    h.remove()

all_features = torch.cat(all_features, dim=0).numpy()
feat_labels = np.array(feat_labels)

print(f"Feature shape: {all_features.shape}")
print(f"Running t-SNE...")

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000)
embeddings = tsne.fit_transform(all_features)

fig, ax = plt.subplots(figsize=(16, 14))
scatter = ax.scatter(embeddings[:, 0], embeddings[:, 1], c=feat_labels, cmap="tab20", s=8, alpha=0.7)
ax.set_title("t-SNE of Test Set Features (Layer 4)", fontsize=14, fontweight="bold")
ax.set_xticks([])
ax.set_yticks([])
plt.colorbar(scatter, ax=ax, label="Class ID", shrink=0.6)
plt.tight_layout()
plt.savefig(EXPERIMENT_DIR / "tsne_features.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Résumé global

In [ ]:
print("=" * 60)
print("RESUME — BCS Determination (ResNet50, Stanford Dogs)")
print("=" * 60)
print(f"Checkpoint:    {CHECKPOINT_PATH.name}")
print(f"Best Epoch:    15")
print(f"Model params:  {sum(p.numel() for p in model.parameters()):,}")
print(f"Train samples: {len(data_module.data_train):,}")
print(f"Val samples:   {len(data_module.data_val):,}")
print(f"Test samples:  {len(data_module.data_test):,}")
print(f"Num classes:   {NUM_CLASSES}")
print("-" * 60)
print(f"Test Top-1 Accuracy:  {accuracy:.2%}")
print(f"Test Top-5 Accuracy:  {top5_accuracy:.2%}")
print(f"ECE (calibration):    {ece:.4f}")
if "val/acc_best" in tb_data:
    print(f"Best Val Acc (TB):    {max(tb_data['val/acc_best']['values']):.2%}")
print(f"Mean confidence:      {max_probs.mean():.2%}")
print(f"Correct predictions:  {correct.sum()} / {len(all_labels)}")
print("=" * 60)